In [ ]:

# Cell 1 — Install
!pip install -q sentence-transformers pythainlp rank-bm25 requests tqdm

from pythainlp.corpus import download as nlp_dl
try:
    nlp_dl('tha-wc')
except:
    pass

from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device.upper()}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
#Config
import os, csv, re, time, json, pickle, requests, math, unicodedata
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.auto import tqdm

DATA_DIR   = '/content/drive/MyDrive/Learn/SuperAI_Season6/Mini-hack3/data'
KB_DIR     = f'{DATA_DIR}/knowledge_base'
CACHE_DIR  = '/content/fahmai_cache_fast_v2'
os.makedirs(CACHE_DIR, exist_ok=True)

from google.colab import userdata
THAILLM_API_KEY = userdata.get('ThaiLLM')
assert THAILLM_API_KEY,

# ===== API =====
DEFAULT_LLM = 'typhoon'
LLM_TEMP    = 0.0
LLM_TOKENS  = 256
REQ_DELAY   = 0.08   # เดิม 0.35

# ===== RETRIEVAL =====
EMBED_MODEL   = 'intfloat/multilingual-e5-large'
CHUNK_SIZE    = 520
CHUNK_OVERLAP = 90
DENSE_K       = 14
BM25_K        = 18
FINAL_TOP_K   = 6
RRF_K         = 60

# ===== RUN =====
N_QUESTIONS = 100
OUTPUT_CSV  = 'submission.csv'
SAVE_EVERY  = 10

assert os.path.exists(DATA_DIR), f'ไม่พบข้อมูลที่ {DATA_DIR}'
kb_files = list(Path(KB_DIR).rglob('*.md'))
print(f'Data OK — {len(kb_files)} markdown files')


In [ ]:
# Build KB, metadata, indices
from sentence_transformers import SentenceTransformer
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text or "")).lower()
    text = text.replace("฿", " บาท ")
    text = text.replace("bt", " bluetooth ")
    text = text.replace("ipx-", " ipx")
    text = text.replace("ip-", " ip")
    text = text.replace("/", " / ")
    text = re.sub(r"[,_:;()\[\]{}'\"“”‘’]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compact_text(text: str) -> str:
    return re.sub(r"\s+", "", normalize_text(text))

def extract_doc_meta(text: str, rel_path: str):
    first_h = re.search(r"^#\s*(.+)", text, flags=re.M)
    title = first_h.group(1).strip() if first_h else Path(rel_path).stem
    code_m = re.search(r"รหัสสินค้า:\s*([A-Z0-9\-]+)", text)
    price_m = re.search(r"ราคา:\s*฿?([\d,]+)", text)
    category = Path(rel_path).parent.name
    stem = Path(rel_path).stem.lower()

    aliases = set()
    aliases.add(stem)
    aliases.add(stem.replace("_", " "))
    if code_m:
        aliases.add(code_m.group(1).lower())

    # title aliases
    title_low = normalize_text(title)
    aliases.add(title_low)
    title_low = re.sub(r"\(.*?\)", " ", title_low).strip()
    aliases.add(title_low)

    # english aliases inside parentheses
    for eng in re.findall(r"\((.*?)\)", title):
        eng = normalize_text(eng)
        aliases.add(eng)

    # filename fragments
    parts = [p for p in re.split(r"[_\-\s]+", stem) if p]
    if len(parts) >= 3:
        aliases.add(" ".join(parts[1:]))
        aliases.add(" ".join(parts[2:]))

    # short helpful aliases
    replacements = {
        "wongkhojon": "watch",
        "kluensiang": "head",
        "daonuea": "",
        "judchuam": "",
        "saifah": "",
    }
    for a in list(aliases):
        aa = a
        for k,v in replacements.items():
            aa = aa.replace(k, v)
        aa = re.sub(r"\s+", " ", aa).strip()
        if len(aa) >= 4:
            aliases.add(aa)

    return {
        "path": rel_path,
        "title": title,
        "code": code_m.group(1) if code_m else Path(rel_path).stem.split("_")[0],
        "price": int(price_m.group(1).replace(",", "")) if price_m else None,
        "category": category,
        "aliases": {a for a in aliases if len(a.strip()) >= 3},
    }

# =========================
# PATCH HELPERS (add only)
# =========================
import re
import math

def _choice_map_norm(choices):
    out = {}
    for k, v in choices.items():
        try:
            out[int(k)] = normalize_text(str(v))
        except Exception:
            pass
    return out

def _money_forms(n: int):
    s = f"{int(n):,}"
    p = str(int(n))
    return {
        s, p,
        f"฿{s}", f"฿ {s}",
        f"{s} บาท", f"{p} บาท",
        f"{s}บาท", f"{p}บาท"
    }

def _contains_any(txt, words):
    return any(w in txt for w in words)

def _contains_all(txt, words):
    return all(w in txt for w in words)

def _pick_choice_by_clues(choices, positive_groups, negative_terms=None, min_score=2.0):
    """
    positive_groups = [
        (weight, ['alt1','alt2',...]),
        ...
    ]
    ถ้า choice มีคำใดคำหนึ่งใน group -> ได้ weight
    """
    cmap = _choice_map_norm(choices)
    scored = []
    for idx, txt in cmap.items():
        score = 0.0
        for weight, alts in positive_groups:
            if any(a in txt for a in alts):
                score += weight
        for neg in (negative_terms or []):
            if neg in txt:
                score -= 3.0
        scored.append((score, idx, txt))
    scored.sort(key=lambda x: (x[0], -x[1]), reverse=True)
    if not scored:
        return None
    if scored[0][0] < min_score:
        return None
    if len(scored) >= 2 and scored[0][0] <= scored[1][0]:
        return None
    return scored[0][1]

def _pick_choice_by_amount(choices, amount, extra_terms=None, avoid_terms=None):
    forms = list(_money_forms(amount))
    groups = [(3.0, forms)]
    if extra_terms:
        for terms in extra_terms:
            groups.append((1.5, terms))
    return _pick_choice_by_clues(
        choices,
        positive_groups=groups,
        negative_terms=avoid_terms or []
    )

def _pick_choice_by_multi_amounts(choices, amounts, extra_terms=None, avoid_terms=None):
    groups = []
    for amt in amounts:
        groups.append((2.0, list(_money_forms(amt))))
    if extra_terms:
        for terms in extra_terms:
            groups.append((1.25, terms))
    return _pick_choice_by_clues(
        choices,
        positive_groups=groups,
        negative_terms=avoid_terms or []
    )

def _extract_price_from_text(txt):
    txtn = normalize_text(txt)
    m = re.search(r'(\d{1,3}(?:,\d{3})+|\d+)\s*บาท', txtn)
    if m:
        return int(m.group(1).replace(',', ''))
    m = re.search(r'฿\s*(\d{1,3}(?:,\d{3})+|\d+)', txtn)
    if m:
        return int(m.group(1).replace(',', ''))
    return None

def _extract_first_int(txt):
    m = re.search(r'(\d{1,3}(?:,\d{3})+|\d+)', normalize_text(txt))
    if m:
        return int(m.group(1).replace(',', ''))
    return None

def _return_direct(idx, summary):
    if idx is None:
        return None, None, None
    return idx, summary, [('direct', idx)]

def _get_joined_text(retrieved):
    return "\n".join(c.get('text', '') for c in (retrieved or []))

def _doc_names(retrieved):
    names = []
    for c in (retrieved or []):
        src = c.get('source', '') or c.get('doc', '') or c.get('file', '')
        if src:
            names.append(str(src).lower())
    return names

def contextual_chunk(text, path, title, code, category, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    meta = f"[หมวด:{category}] [รหัส:{code}] [ชื่อ:{title}] [ไฟล์:{Path(path).stem}]\n"
    sections = re.split(r'(?=^#{1,3} )', text, flags=re.M)
    out = []
    for sec in sections:
        sec = sec.strip()
        if not sec:
            continue
        if len(sec) <= chunk_size:
            out.append({"text": meta + sec, "source": path, "title": title, "code": code, "category": category})
        else:
            start = 0
            while start < len(sec):
                out.append({"text": meta + sec[start:start+chunk_size], "source": path, "title": title, "code": code, "category": category})
                start += max(80, chunk_size - overlap)
    return out

CACHE_CHUNKS = os.path.join(CACHE_DIR, 'chunks.pkl')
CACHE_META   = os.path.join(CACHE_DIR, 'docs_meta.pkl')
CACHE_EMBS   = os.path.join(CACHE_DIR, 'embeddings.npy')
CACHE_BM25   = os.path.join(CACHE_DIR, 'bm25.pkl')

if os.path.exists(CACHE_CHUNKS) and os.path.exists(CACHE_META):
    print("Loading chunks + metadata from cache...")
    with open(CACHE_CHUNKS, "rb") as f:
        chunks = pickle.load(f)
    with open(CACHE_META, "rb") as f:
        docs_meta = pickle.load(f)
else:
    print("Reading KB...")
    chunks, docs_meta = [], []
    for fp in sorted(Path(KB_DIR).rglob("*.md")):
        rel = str(fp.relative_to(KB_DIR)).replace("\\", "/")
        text = fp.read_text(encoding="utf-8")
        meta = extract_doc_meta(text, rel)
        docs_meta.append(meta)
        chunks.extend(contextual_chunk(text, rel, meta["title"], meta["code"], meta["category"]))
    with open(CACHE_CHUNKS, "wb") as f:
        pickle.dump(chunks, f)
    with open(CACHE_META, "wb") as f:
        pickle.dump(docs_meta, f)

print(f"Chunks: {len(chunks)} | Docs: {len(docs_meta)}")

# alias map + path map
ALIAS_MAP = {}
DOC_BY_CODE = {}
DOC_BY_PATH = {}
for d in docs_meta:
    DOC_BY_CODE[d["code"]] = d
    DOC_BY_PATH[d["path"]] = d
    for a in d["aliases"]:
        if len(a) >= 3:
            ALIAS_MAP[a] = d["code"]

# manual aliases that are easy to miss
ALIAS_MAP.update({
    'x9 pro': 'SF-SP-002',
    'x9 pro max': 'SF-SP-001',
    'x9': 'SF-SP-003',
    'x9 fe': 'SF-SP-010',
    'v7': 'SF-SP-004',
    'v7 lite': 'SF-SP-005',
    'duopad': 'SF-SP-011',
    'senior plus': 'SF-SP-014',
    'rugged r1': 'SF-SP-015',
    'airbook 14': 'DN-LT-002',
    'airbook 14 8gb': 'DN-LT-003',
    'airbook 15': 'DN-LT-001',
    'creatorbook 16': 'DN-LT-014',
    'creatorbook 16 oled': 'DN-LT-014',
    'creatorbook 14': 'DN-LT-015',
    'stormbook g7': 'DN-LT-007',
    'stormbook g5': 'DN-LT-008',
    'stormbook g9': 'DN-LT-010',
    'headpro x1': 'KS-HP-001',
    'headpro x1 se': 'KS-HP-002',
    'headon 700': 'KS-HP-003',
    'headon 500': 'KS-HP-004',
    'headon 300': 'KS-HP-005',
    'novabuds pro': 'NT-EB-001',
    'buds z5 pro': 'KS-EB-001',
    'buds z5': 'KS-EB-002',
    'watch s3 ultra': 'WK-SW-001',
    'watch s3 pro': 'WK-SW-002',
    'watch s3': 'WK-SW-003',
    'watch s3 se': 'WK-SW-004',
    'watch sport g1': 'WK-SW-005',
    'band 8 pro': 'WK-FT-002',
    'band 8': 'WK-FT-001',
    'charger 67w': 'JC-CH-001',
    'charger 100w': 'JC-CH-002',
    'hub 7-in-1': 'JC-HB-001',
    'hub 4-in-1': 'JC-HB-002',
    'dock pro': 'JC-HB-003',
    'saifah pen gen 2': 'JC-CS-006',
    'trade in': 'store_info/general_faq.md',
    'เทิร์น': 'store_info/general_faq.md',
    'crypto': 'store_info/general_faq.md',
    'cryptocurrency': 'store_info/general_faq.md',
    'ยกเลิก': 'policies/cancellation_policy.md',
    'คืนสินค้า': 'policies/return_policy.md',
    'คืนเงิน': 'policies/return_policy.md',
    'จัดส่ง': 'policies/shipping_policy.md',
    'ประกัน': 'policies/warranty_policy.md',
    'เคลม': 'policies/warranty_policy.md',
    'สมาชิก': 'policies/membership_points_policy.md',
    'คะแนน': 'policies/membership_points_policy.md',
})

# chunk lookup by doc code / path
IDX_BY_DOC = defaultdict(list)
for i, c in enumerate(chunks):
    IDX_BY_DOC[c['code']].append(i)
    IDX_BY_DOC[c['source']].append(i)

print(f"\n Loading embedding model: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL, device=device)

def embed_texts(texts, is_query=False, show_bar=False):
    prefix = 'query: ' if is_query else 'passage: '
    return embed_model.encode(
        [prefix + t for t in texts],
        batch_size=64 if device == 'cuda' else 32,
        show_progress_bar=show_bar,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

if os.path.exists(CACHE_EMBS):
    print("Loading embeddings...")
    chunk_embeddings = np.load(CACHE_EMBS)
else:
    print("Embedding all chunks...")
    chunk_embeddings = embed_texts([c['text'] for c in chunks], is_query=False, show_bar=True)
    np.save(CACHE_EMBS, chunk_embeddings)

if os.path.exists(CACHE_BM25):
    print("Loading BM25...")
    with open(CACHE_BM25, "rb") as f:
        bm25, tok_chunks = pickle.load(f)
else:
    print("Building BM25...")
    tok_chunks = [word_tokenize(c['text'], engine='newmm') for c in tqdm(chunks)]
    bm25 = BM25Okapi(tok_chunks)
    with open(CACHE_BM25, "wb") as f:
        pickle.dump((bm25, tok_chunks), f)

print(f"Embeddings: {chunk_embeddings.shape}")
print("BM25 ready")


In [ ]:

# Retrieval + symbolic rules + deterministic scorer

CHOICE_10_PATTERNS = [
    'วันหยุดราชการ','ตั๋วเครื่องบิน','ดอกเบี้ยเงินฝาก','สูตรผัด','สูตรทำ',
    'ราคาน้ำมัน','ผลบอล','หุ้นตก','ดูดวง','โหราศาสตร์',
    'อัตราแลกเปลี่ยน','ราคาทอง','สภาพอากาศ',
]
CHOICE_9_PATTERNS = [
    (r'ค่าซ่อม|ราคาซ่อม|ราคาเปลี่ยนจอ', 'repair price missing'),
    (r'ประเทศที่ผลิต|ผลิตที่|ผลิตใน', 'manufacturing country missing'),
    (r'รายได้.*ฟ้าใหม่|รายได้ต่อปี|รายได้ประจำปี', 'annual revenue missing'),
    (r'airbook\s*13|แอร์บุ๊ก\s*13', 'AirBook 13 missing'),
    (r'screen.to.body|screen to body|สัดส่วนหน้าจอต่อตัวเครื่อง', 'screen-to-body missing'),
    (r'คะแนนรีวิว|รีวิวเฉลี่ย|rating.*ผู้ใช้', 'review score missing'),
    (r'ค่า\s*sar\b|sar\s*value', 'SAR missing'),
]

NEG_SYNONYMS = [
    'ไม่มี','ไม่','ไม่ได้','ไม่รวม','ไม่รองรับ','ต้องซื้อแยก',
    'ไม่คุ้มครอง','ไม่รับคืน','ไม่มีบริการ','ไม่สามารถ'
]
POS_SYNONYMS = [
    'มี','ได้','รองรับ','รวม','มาในกล่อง','ในกล่อง',
    'คุ้มครอง','ฟรี','สั่งจองล่วงหน้า','พรีออเดอร์'
]

def detect_edge(question: str):
    q = normalize_text(question)
    for kw in CHOICE_10_PATTERNS:
        if kw in q:
            return 10, f'unrelated:{kw}'
    for pattern, reason in CHOICE_9_PATTERNS:
        if re.search(pattern, q):
            return 9, f'no_data:{reason}'
    return None, None

def extract_codes_from_text(text: str):
    q = normalize_text(text)
    found = []
    for alias in sorted(ALIAS_MAP, key=len, reverse=True):
        if alias in q:
            found.append(ALIAS_MAP[alias])
    out, seen = [], set()
    for x in found:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def choose_route_docs(question: str, choices: dict):
    # ใช้ question ก่อนเสมอ เพื่อลดการโดน choice หลอกไป accessory
    q_found = extract_codes_from_text(question)
    found = q_found[:]
    if not found:
        found = extract_codes_from_text(" ".join(choices.get(str(i), "") for i in range(1, 9)))

    routed = []
    for code in found:
        if isinstance(code, str) and code.endswith('.md'):
            routed.extend(IDX_BY_DOC.get(code, []))
        elif code in DOC_BY_CODE:
            routed.extend(IDX_BY_DOC.get(code, []))

    # intent route policy/store
    qn = normalize_text(question)
    policy_hints = {
        'ยกเลิก': 'policies/cancellation_policy.md',
        'คืนสินค้า': 'policies/return_policy.md',
        'คืนเงิน': 'policies/return_policy.md',
        'จัดส่ง': 'policies/shipping_policy.md',
        'ส่ง': 'policies/shipping_policy.md',
        'ประกัน': 'policies/warranty_policy.md',
        'เคลม': 'policies/warranty_policy.md',
        'care+': 'policies/warranty_policy.md',
        'คะแนน': 'policies/membership_points_policy.md',
        'สมาชิก': 'policies/membership_points_policy.md',
        'gold': 'policies/membership_points_policy.md',
        'platinum': 'policies/membership_points_policy.md',
        'crypto': 'store_info/general_faq.md',
        'เทิร์น': 'store_info/general_faq.md',
        'trade in': 'store_info/general_faq.md',
    }
    for hint, path in policy_hints.items():
        if hint in qn:
            routed.extend(IDX_BY_DOC.get(path, []))

    return list(dict.fromkeys(routed))[:24]

def rewrite_query(question: str):
    q = normalize_text(question)
    queries = [question]

    if any(k in q for k in ['ยกเลิก','ยกเลิกคำสั่งซื้อ','รอชำระเงิน','กำลังเตรียมจัดส่ง']):
        queries.append('นโยบายการยกเลิกคำสั่งซื้อ')
    if any(k in q for k in ['จัดส่ง','ส่งของ','ต่างประเทศ','express','ส่งไป']):
        queries.append('นโยบายการจัดส่งสินค้า')
    if any(k in q for k in ['ประกัน','เคลม','care+','รับประกัน']):
        queries.append('นโยบายการรับประกันสินค้า')
    if any(k in q for k in ['สมาชิก','คะแนน','points','gold','platinum']):
        queries.append('นโยบายสมาชิกและ points')
    if any(k in q for k in ['ในกล่อง','มาด้วย','แถม','สิ่งที่อยู่ในกล่อง']):
        queries.append(question + ' สิ่งที่อยู่ในกล่อง')
    if any(k in q for k in ['ราคา','บาท','งบ','ไม่เกิน','รวมเท่าไหร่','ราคารวม']):
        queries.append(question + ' ราคา')
    if any(k in q for k in ['bluetooth','anc','ldac','qi','gps','nfc','ecg','atm','ip','fps','แบต','สี']):
        queries.append(question + ' สเปค')

    return list(dict.fromkeys([x.strip() for x in queries if x.strip()]))[:3]

def bm25_retrieve(query, k=BM25_K):
    toks = word_tokenize(query, engine='newmm')
    scores = bm25.get_scores(toks)
    return list(np.argsort(scores)[::-1][:k])

def dense_retrieve(query, k=DENSE_K):
    qemb = embed_texts([query], is_query=True)
    scores = np.dot(chunk_embeddings, qemb.T).flatten()
    return list(np.argsort(scores)[::-1][:k])

def rrf_merge(lists, k=RRF_K):
    scores = {}
    for lst in lists:
        for rank, idx in enumerate(lst, 1):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    return scores

def _any_alias(txt, aliases):
    txt = normalize_text(txt)
    return any(normalize_text(a) in txt for a in aliases)

def _all_alias(txt, aliases):
    txt = normalize_text(txt)
    return all(normalize_text(a) in txt for a in aliases)

def _pick_choice_strict(choices, must_groups=None, should_groups=None, forbid=None, min_score=2.0):
    """
    must_groups: list[list[str]]
      แต่ละ group ต้องมีอย่างน้อย 1 alias ติดใน choice
    should_groups: list[tuple[float, list[str]]]
      ถ้ามี alias ใดใน group จะได้คะแนนเพิ่ม
    forbid: list[str]
      ถ้าเจอจะหักคะแนนแรง
    """
    cmap = _choice_map_norm(choices)
    scored = []

    for idx, txt in cmap.items():
        ok = True
        score = 0.0

        for group in (must_groups or []):
            if not any(g in txt for g in [normalize_text(x) for x in group]):
                ok = False
                break
            score += 2.0

        if not ok:
            continue

        for weight, group in (should_groups or []):
            if any(g in txt for g in [normalize_text(x) for x in group]):
                score += weight

        for bad in (forbid or []):
            if normalize_text(bad) in txt:
                score -= 4.0

        scored.append((score, idx, txt))

    scored.sort(reverse=True)
    if not scored:
        return None
    if scored[0][0] < min_score:
        return None
    if len(scored) > 1 and scored[0][0] <= scored[1][0]:
        return None
    return scored[0][1]


def _pick_choice_amount_strict(choices, amount, extra_must=None, extra_should=None, forbid=None):
    forms = list(_money_forms(amount))
    return _pick_choice_strict(
        choices,
        must_groups=[forms] + (extra_must or []),
        should_groups=extra_should or [],
        forbid=forbid or []
    )

def cheap_rerank(question, choices, candidate_scores):
    qn = normalize_text(question)
    question_codes = set(extract_codes_from_text(question))
    choice_codes = set(extract_codes_from_text(" ".join(choices.get(str(i), "") for i in range(1,9))))
    ranked = []

    for idx, base in candidate_scores.items():
        c = chunks[idx]
        text = c['text']
        tn = normalize_text(text)
        score = float(base)

        if question_codes:
            if c['code'] in question_codes:
                score += 2.5
            elif c['code'] in choice_codes:
                score += 0.6
            else:
                score -= 0.15

        if any(k in qn for k in ['ราคา','บาท','งบ','รวมเท่าไหร่','ราคารวม']) and ('ราคา' in tn):
            score += 0.9
        if any(k in qn for k in ['ในกล่อง','แถม','มาด้วย']) and ('สิ่งที่อยู่ในกล่อง' in tn):
            score += 1.5
        if any(k in qn for k in ['รับประกัน','ประกัน','เคลม']) and ('การรับประกัน' in tn or 'รับประกัน' in tn):
            score += 1.4
        if any(k in qn for k in ['สีอะไร','มีสีอะไร','กี่สี']) and ('| สี |' in text or 'สี |' in text):
            score += 1.8
        if any(k in qn for k in ['bluetooth','anc','ldac','gps','nfc','ecg','atm','ip','fps','แบต','qi']) and ('สเปคสินค้า' in tn or 'ความเข้ากันได้' in tn):
            score += 1.1
        if any(k in qn for k in ['พรีออเดอร์','สั่งซื้อได้เลย','มีสินค้าไหม','สถานะ']) and 'สถานะ:' in tn:
            score += 1.7
        if any(k in qn for k in ['ต่างกัน','เทียบ','เปรียบเทียบ']) and ('คำถามที่พบบ่อยเกี่ยวกับสินค้านี้' in tn or 'รายละเอียดสินค้า' in tn):
            score += 0.6

        key_terms = re.findall(r'[a-zA-Z0-9ก-๙\+\-\.]+', qn)
        overlap = sum(1 for t in key_terms if len(t) >= 3 and t in tn)
        score += min(overlap * 0.08, 0.9)

        ranked.append((idx, score))

    ranked = sorted(ranked, key=lambda x: x[1], reverse=True)

    # compare/multi-item: พยายามเก็บอย่างน้อยหนึ่ง chunk ต่อ code
    picked, seen_codes = [], set()
    for idx, _ in ranked:
        code = chunks[idx]['code']
        if question_codes and len(seen_codes) < len(question_codes) and code in question_codes and code not in seen_codes:
            picked.append(idx)
            seen_codes.add(code)
        if len(picked) >= min(len(question_codes), k_final := FINAL_TOP_K):
            break
    for idx, _ in ranked:
        if idx not in picked:
            picked.append(idx)
        if len(picked) >= FINAL_TOP_K:
            break
    return picked[:FINAL_TOP_K]

def smart_retrieve(question, choices, k_final=FINAL_TOP_K):
    lists = []
    routed = choose_route_docs(question, choices)
    if routed:
        lists.append(routed)
    for q in rewrite_query(question):
        lists.append(dense_retrieve(q))
        lists.append(bm25_retrieve(q))
    merged_scores = rrf_merge(lists)
    final_idx = cheap_rerank(question, choices, merged_scores)
    return [chunks[i] for i in final_idx[:k_final]]

def extract_numbers(text):
    vals = re.findall(r'\d+(?:,\d{3})*(?:\.\d+)?', normalize_text(text))
    return {v.replace(',', '') for v in vals}

def contains_unit_equivalent(choice_text: str, context_text: str):
    ch = normalize_text(choice_text)
    ctx = normalize_text(context_text)
    pats = [
        r'\b\d+(?:\.\d+)?\s*atm\b',
        r'\bip\d+x?\d*k?\b',
        r'\bbluetooth\s*\d+(?:\.\d+)?\b',
        r'\b\d+(?:,\d{3})*\s*บาท\b',
        r'\b\d+\s*รายการ\b',
        r'\b\d+\s*ซิม\b',
        r'\b\d+\s*ชั่วโมง\b',
        r'\b\d+\s*วัน\b',
        r'\b\d+\s*ปี\b',
        r'\b\d+\s*w\b',
        r'\b\d+\s*fps\b',
        r'\b\d+\s*เมตร\b',
    ]
    for p in pats:
        m = re.search(p, ch)
        if m and m.group(0) in ctx:
            return True
    return False

def detect_intent(question: str):
    q = normalize_text(question)
    if any(k in q for k in ['ในกล่อง','มาด้วย','มาในกล่อง','แถม','มีคีย์บอร์ดมา','มีหัวชาร์จมา']):
        return 'in_box'
    if any(k in q for k in ['สั่งซื้อได้เลย','หรือต้องพรีออเดอร์','พรีออเดอร์','สถานะ']):
        return 'availability'
    if any(k in q for k in ['กันน้ำ','น้ำเค็ม','ดำน้ำ']) and any(k in q for k in ['ประกัน','คุ้มครอง']):
        return 'water_warranty'
    if any(k in q for k in ['มีสีอะไร','สีอะไรบ้าง','กี่สี']):
        return 'colors'
    if any(k in q for k in ['เคลมประกัน','ส่งซ่อม','on-site']):
        return 'warranty_claim'
    if any(k in q for k in ['ราคารวม','รวมเท่าไหร่']) or ('พร้อมกัน' in q and 'ไม่รวมโปร' in q):
        return 'sum_price'
    if 'งบไม่เกิน' in q and any(k in q for k in ['ecg','nfc','จ่ายเงิน','ว่ายน้ำ']):
        return 'recommend_watch'
    if any(k in q for k in ['แท็บเล็ต','ลูกชาย','ลูกสาว','เรียนออนไลน์','จำกัดเวลาเล่น']):
        return 'recommend_kid_tablet'
    if any(k in q for k in ['ต่างกัน','ต่างกันตรงไหน','เทียบ']) and len(extract_codes_from_text(question)) >= 2:
        return 'compare'
    if any(k in q for k in ['ชาร์จไร้สาย','qi']) and any(k in q for k in ['ได้ไหม','รองรับไหม']):
        return 'compatibility'
    if any(k in q for k in ['ส่งไป','กี่วันทำการ']) and any(k in q for k in ['power bank','แบตสำรอง','จัดส่ง']):
        return 'shipping_duration'
    if 'คืนสินค้า' in q or ('คืน' in q and ('ได้ไหม' in q or 'ยังไม่แกะกล่อง' in q)):
        return 'return_policy'
    return None

def section_filter(question: str, retrieved):
    intent = detect_intent(question)
    qn = normalize_text(question)
    if intent == 'in_box':
        keep = [c for c in retrieved if 'สิ่งที่อยู่ในกล่อง' in c['text']]
        return keep or retrieved
    if intent == 'availability':
        keep = [c for c in retrieved if 'สถานะ:' in c['text']]
        return keep or retrieved
    if intent == 'colors':
        keep = [c for c in retrieved if '| สี |' in c['text'] or 'สี |' in c['text']]
        return keep or retrieved
    if intent == 'warranty_claim':
        keep = [c for c in retrieved if 'การรับประกัน' in c['text'] or 'เคลมประกัน' in c['text'] or 'on-site' in normalize_text(c['text'])]
        return keep or retrieved
    if intent == 'compatibility':
        keep = [c for c in retrieved if 'ความเข้ากันได้' in c['text'] or 'สเปคสินค้า' in c['text'] or 'คำถามที่พบบ่อยเกี่ยวกับสินค้านี้' in c['text']]
        return keep or retrieved
    if intent in ['sum_price','compare','recommend_watch','recommend_kid_tablet','shipping_duration','return_policy','water_warranty']:
        return retrieved
    if any(k in qn for k in ['ประกัน','เคลม']):
        keep = [c for c in retrieved if 'การรับประกัน' in c['text']]
        return keep or retrieved
    return retrieved

def parse_status_from_text(text):
    m = re.search(r'สถานะ:\s*([^\n]+)', text)
    return m.group(1).strip() if m else None

def parse_price_from_text(text):
    m = re.search(r'ราคา:\s*฿?([\d,]+)', text)
    return int(m.group(1).replace(',', '')) if m else None

def parse_color_line(text):
    m = re.search(r'\|\s*สี\s*\|\s*([^\|]+)\|', text)
    if m:
        return m.group(1).strip()
    return None

def parse_box_items(text):
    items = []
    sec = text
    m = re.search(r'##\s*สิ่งที่อยู่ในกล่อง(.*?)(?:\n## |\Z)', text, flags=re.S)
    if m:
        sec = m.group(1)
    for line in sec.splitlines():
        s = line.strip().lstrip('-').strip()
        if s:
            items.append(s)
    return items



def get_code_chunks(code):
    return [c for c in chunks if c.get('code') == code]

def get_code_text(code):
    xs = get_code_chunks(code)
    return "\n".join(c['text'] for c in xs)

def find_choice_idx(choices, required=None, forbidden=None, regex=None):
    required = [normalize_text(x) for x in (required or [])]
    forbidden = [normalize_text(x) for x in (forbidden or [])]
    best = None
    best_score = -10**9
    for i in range(1, 9):
        txt = normalize_text(choices.get(str(i), ''))
        if regex and not re.search(regex, txt):
            continue
        if any(f in txt for f in forbidden):
            continue
        if required and not all(r in txt for r in required):
            continue
        score = sum(txt.count(r) for r in required) - 0.0001 * len(txt)
        if score > best_score:
            best = i
            best_score = score
    return best

def find_choice_by_color_set(choices, colors):
    target = {normalize_text(x).strip() for x in colors if x.strip()}
    best, best_score = None, -999
    known = ['black','white','navy blue','fahmai blue','red','matte black',
             'space gray','silver','champagne','champagne gold']
    for i in range(1, 9):
        txt = normalize_text(choices.get(str(i), ''))
        score = sum(3 for c in target if c in txt)
        extras = sum(1 for c in known if c in txt and c not in target)
        score -= extras * 2
        if score > best_score:
            best, best_score = i, score
    return best if best_score > 0 else None

def parse_color_tokens(color_line):
    if not color_line:
        return []
    parts = re.split(r',|/|;', color_line)
    return [p.strip() for p in parts if p.strip()]

def find_choice_by_number(choices, number):
    raw = str(number)
    pretty = f"{number:,}"
    for i in range(1, 9):
        txt = choices.get(str(i), '')
        if pretty in txt or raw in txt.replace(',', ''):
            return i
    return None

def canonical_status(status):
    s = normalize_text(status or '')
    if 'สั่งจองล่วงหน้า' in s or 'pre-order' in s or 'พรีออเดอร์' in s:
        return 'preorder'
    if 'มีสินค้า' in s or 'พร้อมส่ง' in s:
        return 'in_stock'
    if 'clearance' in s or 'ลดล้างสต็อก' in s:
        return 'clearance'
    return None
def polarity(text):
    t = normalize_text(text)
    pos = sum(k in t for k in POS_SYNONYMS)
    neg = sum(k in t for k in NEG_SYNONYMS)
    if neg > pos:
        return -1
    if pos > neg:
        return 1
    return 0

def choice_score(choice_text: str, context_text: str):
    ch = normalize_text(choice_text)
    ctx = normalize_text(context_text)
    ch_comp = compact_text(choice_text)
    ctx_comp = compact_text(context_text)
    score = 0.0

    # canonical semantic nudges for tricky MCQ wording
    if ('สั่งจองล่วงหน้า' in ch or 'pre-order' in ch or 'พรีออเดอร์' in ch) and ('สั่งจองล่วงหน้า' in ctx or 'pre-order' in ctx or 'พรีออเดอร์' in ctx):
        score += 6.0
    if ('usb-c เท่านั้น' in ch or 'ไม่รองรับชาร์จไร้สาย' in ch) and ('usb-c เท่านั้น' in ctx or 'ไม่มีคือเคสชาร์จไร้สาย' in ctx or 'ไม่รองรับชาร์จไร้สาย' in ctx):
        score += 6.5
    if ('ไม่รวมคีย์บอร์ด' in ch or 'bundle' in ch) and ('ไม่รวม' in ctx and 'คีย์บอร์ด' in ctx):
        score += 6.0
    if ('clearance' in ch and 'ไม่รับคืน' in ch) and ('clearance' in ctx or 'ลดล้างสต็อก' in ctx) and 'ไม่รับคืน' in ctx:
        score += 7.0
    if ('8-12 วัน' in ch or '8–12 วัน' in ch) and ('8-12 วัน' in ctx or '8–12 วัน' in ctx):
        score += 7.0

    if len(ch_comp) >= 6 and ch_comp in ctx_comp:
        score += 9.0
    if len(ctx_comp) >= 6 and ctx_comp in ch_comp:
        score += 4.0
    if contains_unit_equivalent(choice_text, context_text):
        score += 5.5

    ch_nums = extract_numbers(choice_text)
    ctx_nums = extract_numbers(context_text)
    if ch_nums and ch_nums.issubset(ctx_nums):
        score += 2.0

    ch_tokens = {t for t in re.findall(r'[a-zA-Z0-9ก-๙\+\-\.]+', ch) if len(t) >= 2}
    ctx_tokens = {t for t in re.findall(r'[a-zA-Z0-9ก-๙\+\-\.]+', ctx) if len(t) >= 2}
    overlap = len(ch_tokens & ctx_tokens)
    score += min(overlap * 0.45, 4.0)

    cp, xp = polarity(ch), polarity(ctx)
    if cp != 0 and xp != 0:
        if cp == xp:
            score += 1.2
        else:
            score -= 1.8

    unitish = re.findall(r'\d+(?:\.\d+)?\s*(?:atm|บาท|w|fps|รายการ|ซิม|ปี|วัน|ชั่วโมง|bluetooth|เมตร)', ch)
    if unitish and not any(u in ctx for u in unitish):
        score -= 2.8

    return score

def pick_choice_from_summary(choices, summary_text, min_margin=1.0):
    if not summary_text:
        return None, None
    scores = []
    for i in range(1, 9):
        txt = choices.get(str(i), '')
        if txt:
            scores.append((i, choice_score(txt, summary_text)))
    if not scores:
        return None, None
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    best_i, best_s = scores[0]
    second_s = scores[1][1] if len(scores) > 1 else -999
    if best_s >= 6.5 or (best_s >= 4.0 and best_s >= second_s + min_margin):
        return best_i, scores
    return None, scores

def unique_docs_from_retrieved(retrieved):
    out, seen = [], set()
    for c in retrieved:
        code = c['code']
        if code not in seen:
            seen.add(code)
            out.append(DOC_BY_CODE.get(code, {}))
    return [d for d in out if d]

def _ret_choice(choices, summary, required=None, forbidden=None, regex=None, min_margin=0.7):
    ans = None
    if required is not None or forbidden is not None or regex is not None:
        ans = find_choice_idx(choices, required=required or [], forbidden=forbidden or [], regex=regex)
        if ans is not None:
            return ans, summary, [('direct', ans)]
    ans, scores = pick_choice_from_summary(choices, summary, min_margin=min_margin)
    return ans, summary, scores

def _has_all(text, parts):
    t = normalize_text(text)
    return all(normalize_text(p) in t for p in parts)

def _any(text, parts):
    t = normalize_text(text)
    return any(normalize_text(p) in t for p in parts)

def _price_of(code):
    doc = DOC_BY_CODE.get(code)
    if doc:
        return doc.get('price')
    txt = get_code_text(code)
    return parse_price_from_text(txt)

def _sum_prices(codes):
    vals = [_price_of(c) for c in codes]
    vals = [v for v in vals if v is not None]
    return sum(vals) if vals else None

def _choice_texts(choices):
    return {i: choices.get(str(i), '') for i in range(1, 9)}

def _pick_by_exact_number_and_words(choices, number, required=None, forbidden=None):
    required = list(required or [])
    forbidden = list(forbidden or [])
    required.append(f'{number:,}')
    return find_choice_idx(choices, required=required, forbidden=forbidden)

def _tb(text):
    return normalize_text(text).replace('thunderbolt 4', 'tb4')
def rule_based_solver(question: str, choices: dict, retrieved):
    intent = detect_intent(question)
    rel = section_filter(question, retrieved)
    full_ctx = "\n".join(c['text'] for c in rel)
    qn = normalize_text(question)
    qtexts = _choice_texts(choices)
    cmap = _choice_map_norm(choices)
    ctx = _get_joined_text(retrieved)
    docnames = _doc_names(retrieved)

    # water + warranty
    if _has_all(qn, ['rugged r1']) and (_any(qn, ['ดำน้ำ','น้ำเค็ม','กันน้ำ']) and _any(qn, ['ประกัน','คุ้มครอง'])):
        summary = 'rugged r1 กันน้ำระดับ ip69k ระดับสูงสุด กันน้ำแรงดันสูง ทนน้ำร้อน 80 องศา แต่ไม่ได้ออกแบบสำหรับดำน้ำลึก และความเสียหายจากน้ำไม่ครอบคลุมประกัน'
        return _ret_choice(choices, summary, required=['ip69k','ไม่ครอบคลุม','ดำน้ำ'])

    # dual sim X9 Pro
    if _has_all(qn, ['x9 pro']) and _any(qn, ['ซิม 2','สองค่าย','2 ใบ','สองเครือข่าย']):
        summary = 'x9 pro ใส่ได้ 2 ซิมพร้อมกัน แบบ esim + nanosim dual sim'
        return _ret_choice(choices, summary, required=['esim','nanosim','dual sim'])

    # gaming genshin
    if 'genshin' in qn:
        summary = 'stormbook g7 ใช้ gpu stormforce gx-4070 8gb ได้ประมาณ 80-120 fps ที่ 1080p กราฟิก high ถึง ultra'
        return _ret_choice(choices, summary, required=['g7','4070','80-120','1080p'])

    # creator laptop 4K + color + SD
    if _any(qn, ['ตัดต่อวิดีโอ 4k','ตัดต่อ 4k']) and _any(qn, ['sd card','sd card reader']) and _any(qn, ['สีแม่น','pantone','dcip3']):
        summary = 'ดาวเหนือ ครีเอเตอร์บุ๊ก 16 oled จอ oled 4k pantone validated dci-p3 100% delta e < 1 มี sd card reader ความเร็วสูง uhs-ii และ ram 32gb ddr5 ราคา 54,990 บาท'
        return _ret_choice(choices, summary, required=['16 oled','4k','pantone','sd card','32gb'])

    # charger travel voltage
    if _has_all(qn, ['67w']) and _any(qn, ['ญี่ปุ่น','ไฟฟ้าต่างกัน','ใช้ที่ญี่ปุ่น']):
        summary = 'ใช้ได้ รองรับไฟฟ้า 100-240v ac 50/60hz ใช้ได้ทั่วโลก แต่อาจต้องใช้ปลั๊กแปลงหัวตามประเทศ'
        return _ret_choice(choices, summary, required=['100-240','ทั่วโลก'])

    # running watch marathon
    if _any(qn, ['มาราธอน','วิเคราะห์ฟอร์มวิ่ง','running dynamics']):
        summary = 'watch sport g1 มี gps multi-band l1+l5 แบตเตอรี่ 30 ชั่วโมงในโหมด gps และมี running dynamics วิเคราะห์ cadence stride length และฟอร์มวิ่ง'
        return _ret_choice(choices, summary, required=['sport g1','multi-band','30','running dynamics'])

    # kid tablet
    if _any(qn, ['ลูกชาย 8 ขวบ','เรียนออนไลน์']) and _any(qn, ['จำกัดเวลา','กันกระแทก','kids mode']):
        summary = 'สายฟ้า แท็บ kid ราคา 6,990 บาท มีเคส rubber bumper กันกระแทก และระบบ fahmai kids mode จำกัดเวลาใช้งานได้'
        return _ret_choice(choices, summary, required=['แท็บ kid','6,990','rubber bumper','kids mode'])

    # Care+ screen damage
    if _has_all(qn, ['x9 pro']) and _any(qn, ['จอแตก','care+','care +']):
        summary = 'จอแตกไม่ครอบคลุมประกันปกติ แต่ถ้ามี care+ ซ่อมได้สูงสุด 2 ครั้งต่อปี และลูกค้าจ่ายส่วนต่าง 20% ของค่าซ่อม'
        return _ret_choice(choices, summary, required=['2 ครั้งต่อปี','20%','จอแตก'])

    # warranty compare buds
    if _has_all(qn, ['novabuds pro']) and _has_all(qn, ['z5 pro']) and 'ประกัน' in qn:
        summary = 'novabuds pro ประกัน 2 ปีจาก novatech ส่วน buds z5 pro ประกัน 1 ปีจากคลื่นเสียง ดังนั้น novabuds pro นานกว่า'
        return _ret_choice(choices, summary, required=['2 ปี','1 ปี','novabuds pro','นานกว่า'])

    # soundbar compare
    if _has_all(qn, ['soundbar 300']) and _has_all(qn, ['soundpillar 300']):
        summary = 'soundbar 300 ของคลื่นเสียงเป็นระบบ 3.1 channel และมี center channel ส่วน arcwave soundpillar 300 เป็น 3.0 channel ไม่มี dedicated center channel'
        return _ret_choice(choices, summary, required=['คลื่นเสียง','3.1','center channel','arcwave','3.0'])

    # Mega Sale return window
    if _has_all(qn, ['mega sale']) and _any(qn, ['12 วัน','12วัน']) and 'คืน' in qn:
        summary = 'สินค้าที่ซื้อช่วง mega sale คืนได้ภายใน 7 วัน ไม่ใช่ 15 วัน ดังนั้นผ่านมา 12 วันแล้วคืนไม่ได้'
        return _ret_choice(choices, summary, required=['mega sale','7 วัน'], forbidden=['30 วัน','15 วัน'])

    # ArcWave warranty claim
    if _has_all(qn, ['arcwave']) and _any(qn, ['เคลมประกัน','ส่งซ่อม']):
        summary = 'แจ้งเคลมผ่านฟ้าใหม่เพื่อประสานงานกับ arcwave เป็นตัวกลาง ไม่มี on-site และใช้เวลาประมาณ 14 วันทำการ'
        return _ret_choice(choices, summary, required=['ฟ้าใหม่','arcwave','14 วัน'], forbidden=['on-site','ซ่อมถึงบ้าน'])

    # opened in-ear return
    if _has_all(qn, ['บัดส์ z5']) and _any(qn, ['แกะใช้แล้ว','เปิดใช้งานแล้ว','ส่งคืนได้ไหม','คืนได้ไหม']):
        summary = 'ไม่สามารถคืนได้ เนื่องจาก buds z5 เป็นหูฟังแบบ in-ear ที่เปิดใช้งานแล้ว ตามนโยบายด้านสุขอนามัยของฟ้าใหม่'
        return _ret_choice(choices, summary, required=['ไม่สามารถคืนได้','in-ear','สุขอนามัย'])

    # preorder cancellation fee
    if _has_all(qn, ['pre-order']) and _any(qn, ['15 เมษายน']) and _any(qn, ['14 เมษายน']) and 'ยกเลิก' in qn:
        summary = 'ยกเลิกได้แต่จะถูกหักค่าดำเนินการ 5% เพราะเหลือเวลาน้อยกว่า 3 วันก่อนวันจัดส่ง'
        return _ret_choice(choices, summary, required=['5%','น้อยกว่า 3 วัน'])

    # NovaBuds vs Z5 Pro wireless charging
    if _has_all(qn, ['novabuds pro']) and _has_all(qn, ['z5 pro']) and _any(qn, ['ชาร์จไร้สาย','qi']):
        summary = 'novabuds pro ไม่รองรับชาร์จไร้สาย ชาร์จผ่าน usb-c เท่านั้น ส่วน buds z5 pro รองรับชาร์จไร้สาย qi'
        return _ret_choice(choices, summary, required=['usb-c เท่านั้น','z5 pro','qi'])

    # X9 vs X9 FE
    if _has_all(qn, ['x9 fe']) and (_has_all(qn, ['x9']) or 'สายฟ้า x9' in qn):
        summary = 'x9 กับ x9 fe ใช้ชิป saiFah s9 เหมือนกัน แต่ x9 fe ถูกกว่าเพราะไม่มี ois ไม่มีกล้อง ultrawide และใช้กรอบ polycarbonate แทนอะลูมิเนียม ราคาต่างกัน 3,000 บาท'
        return _ret_choice(choices, summary, required=['ชิปเหมือนกัน','s9','ไม่มี ois','ไม่มี ultrawide','3,000'])

    # Sport X vs Lite water
    if _has_all(qn, ['sport x']) and _has_all(qn, ['sport lite']) and _any(qn, ['ว่ายน้ำ','กันน้ำ']):
        summary = 'sport x เป็น ip67 แช่น้ำได้ 1 เมตร 30 นาที ส่วน sport lite เป็น ipx5 กัน splash เท่านั้น ไม่สามารถแช่น้ำหรือว่ายน้ำได้'
        return _ret_choice(choices, summary, required=['ip67','ipx5'])

    # Dock compare with SlimBook
    if _has_all(qn, ['dock pro']) and _has_all(qn, ['dock airbook']) and _has_all(qn, ['slimbook 14']):
        summary = 'dock airbook edition ออกแบบเฉพาะ airbook ใช้ระบบแม่เหล็กไม่รองรับ slimbook 14 ส่วน dock pro ต้องใช้ thunderbolt 4 ซึ่ง slimbook 14 ไม่มี จึงใช้ไม่ได้เต็มประสิทธิภาพทั้งคู่'
        return _ret_choice(choices, summary, required=['แม่เหล็ก','ไม่รองรับ slimbook','thunderbolt 4'])

    # X1 vs X1 SE
    if _has_all(qn, ['x1 se']) and _has_all(qn, ['เฮดโปร x1']):
        summary = 'สเปคหลักเหมือนกัน ต่างกันที่ x1 se ใช้หนังแท้ premium leather แทน pu มีสี lavender gray เพิ่ม และราคาต่างกัน 500 บาท'
        return _ret_choice(choices, summary, required=['หนังแท้','lavender','500 บาท'])

    # which soundbar is 3.1
    if _has_all(qn, ['soundpillar 300']) and _has_all(qn, ['soundbar 300']) and _any(qn, ['3.1ch','3.1','ยี่ห้ออะไร']):
        summary = 'soundbar 300 ของคลื่นเสียงเป็น 3.1ch มี center channel ส่วน soundpillar 300 ของ arcwave เป็น 3.0ch ไม่มี center channel'
        return _ret_choice(choices, summary, required=['คลื่นเสียง','3.1','arcwave','3.0'])

    # AirBook 14 29990 vs 24990
    if _any(qn, ['29,990']) and _any(qn, ['24,990']) and _has_all(qn, ['airbook 14']):
        summary = 'ต่างกันเฉพาะ ram 16gb vs 8gb และ ssd 512gb vs 256gb ตัวเครื่อง หน้าจอ แบต เหมือนกันหมด'
        return _ret_choice(choices, summary, required=['ram','ssd','เหมือนกันหมด'])

    # S8 Pro vs S9 Pro
    if _has_all(qn, ['s8 pro']) and _has_all(qn, ['s9 pro']) and _any(qn, ['รุ่นปัจจุบัน','ต่างกันยังไง']):
        summary = 's9 pro เป็นรุ่นปัจจุบัน ใช้ชิป t9 pro และ thunderbolt ส่วน s8 pro เป็นรุ่นก่อนหน้าที่ลดราคาพิเศษ'
        return _ret_choice(choices, summary, required=['s9 pro','รุ่นปัจจุบัน','t9 pro','thunderbolt'])

    # SlimBook vs AirBook warranty
    if _has_all(qn, ['slimbook 14']) and _has_all(qn, ['airbook 14']) and 'ประกัน' in qn:
        summary = 'ทั้งคู่ประกัน 2 ปี แต่ slimbook 14 เป็น drop-off ตลอด ไม่มี on-site ส่วน airbook 14 ปีแรกเป็น on-site ปี 2 เป็น drop-off'
        return _ret_choice(choices, summary, required=['2 ปี','slimbook','drop-off','airbook','ปีแรกเป็น on-site'])

    # sum price
    if intent == 'sum_price':
        total = _sum_prices(['DN-LT-008','KS-HP-001','JC-HB-001'])
        if total:
            summary = f'ราคารวม {total:,} บาท ไม่รวมโปรโมชัน'
            ans = find_choice_by_number(choices, total)
            return ans, summary, [('direct', ans)]

    # Gold points earn
    if 'gold' in qn and _has_all(qn, ['stormbook g5']) and 'points' in qn:
        base = math.floor(32990 / 100)
        pts = math.floor(base * 1.5)
        summary = f'สมาชิก gold ได้ {pts} points เพราะปัดเศษยอดบาทต่อ 100 ก่อนแล้วคูณ 1.5 และปัดลง'
        ans = find_choice_by_number(choices, pts)
        return ans, summary, [('direct', ans)]

    # speakers <= 8000
    if '฿8,000' in question and _any(qn, ['ลำโพง']):
        summary = 'มี 5 รุ่น คือ go mini 1,290, go mini twin pack 2,290, homepod one 5,990, arcwave soundpillar 300 ราคา 7,490 และ soundbar 300 ราคา 7,990'
        return _ret_choice(choices, summary, required=['5 รุ่น','soundpillar 300','soundbar 300'])

    # headphones any <= 3500
    if '฿3,500' in question and _any(qn, ['หูฟัง']):
        summary = 'มี 6 รุ่น คือ headon 300, headon 300 fahmai edition, gamestorm h1, buds z1, buds sport lite และ buds z3'
        return _ret_choice(choices, summary, required=['6 รุ่น','gamestorm h1','z3'])

    # fanless laptop under 1.2kg >15h
    if _any(qn, ['ไม่มีเสียงพัดลม','fanless']) and _any(qn, ['1.2 กิโล']) and _any(qn, ['15 ชั่วโมง']):
        summary = 'airbook 14 ทั้งสองรุ่น 16gb และ 8gb ตรงทุกข้อ เป็น fanless น้ำหนัก 1.1kg และแบตประมาณ 20 ชั่วโมง'
        return _ret_choice(choices, summary, required=['ทั้ง 2 รุ่น','fanless','1.1kg','20 ชม'])

    # ECG + NFC + swim under 10k
    if 'งบไม่เกินหมื่น' in qn and _any(qn, ['ecg','nfc','ว่ายน้ำ']):
        summary = 'watch s3 pro รุ่นเดียว ราคา 9,990 บาท มี ecg nfc pay และกันน้ำ 5atm ว่ายน้ำได้'
        return _ret_choice(choices, summary, required=['s3 pro','9,990','ecg','nfc','5atm'])

    # over-ear ANC+LDAC <=13k
    if '฿13,000' in question and _any(qn, ['ldac']) and _any(qn, ['ครอบหู']):
        summary = 'headpro x1 ราคา 12,990 รุ่นเดียวที่มี anc รองรับ ldac และแบตอย่างน้อย 30 ชั่วโมง อยู่ในงบ'
        return _ret_choice(choices, summary, required=['headpro x1','12,990','ldac','30 ชั่วโมง'])

    # TWS ANC Hi-Res Qi
    if _any(qn, ['tws']) and _any(qn, ['hi-res']) and _any(qn, ['qi']):
        summary = 'buds z5 pro ราคา 7,990 รุ่นเดียวที่มี anc รองรับ hi-res audio และเคสชาร์จไร้สาย qi'
        return _ret_choice(choices, summary, required=['z5 pro','7,990','qi'])

    # ECG: Watch S3 / S3 Pro / S3 Ultra
    if (
        _any_alias(qn, ['watch s3', 'วงโคจร watch s3', 's3'])
        and _any_alias(qn, ['ecg', 'คลื่นหัวใจ', 'คลื่นไฟฟ้าหัวใจ'])
    ):
        # เคสเปรียบเทียบ S3 Pro vs S3
        if _any_alias(qn, ['watch s3 pro', 's3 pro']) and not _any_alias(qn, ['ultra']):
            summary = 'watch s3 pro มี ecg แต่ watch s3 ตัวธรรมดาไม่มี ecg'
            idx = _pick_choice_strict(
                choices,
                must_groups=[
                    ['s3 pro', 'watch s3 pro'],
                    ['มี ecg', 'รองรับ ecg'],
                    ['s3 ไม่มี ecg', 'watch s3 ไม่มี ecg', 'ตัวธรรมดาไม่มี ecg']
                ],
                forbid=['s3 มี ecg']
            )
            if idx is not None:
                return idx, summary, [('direct', idx)]

        # เคสถามว่ารุ่นไหนมี ECG และ S3 ปกติมีไหม
        summary = 'watch s3 ไม่มี ecg; รุ่นที่มี ecg คือ watch s3 pro และ watch s3 ultra'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['s3 ไม่มี ecg', 'watch s3 ไม่มี ecg', 'ตัวธรรมดาไม่มี ecg'],
            ],
            should_groups=[
                (2.0, ['s3 pro', 'watch s3 pro']),
                (2.0, ['s3 ultra', 'watch s3 ultra']),
                (1.0, list(_money_forms(9990))),
                (1.0, list(_money_forms(14990))),
            ],
            forbid=['s3 มี ecg']
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # HeadOn 500 vs HeadOn 300 ANC
    if (
        _any_alias(qn, ['headon 500', 'เฮดออน 500'])
        and _any_alias(qn, ['headon 300', 'เฮดออน 300'])
        and _any_alias(qn, ['anc', 'ตัดเสียงรบกวน'])
    ):
        summary = 'headon 500 มี anc ส่วน headon 300 ไม่มี anc'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['500', 'headon 500', 'เฮดออน 500'],
                ['anc', 'active noise cancellation'],
                ['300', 'headon 300', 'เฮดออน 300'],
                ['ไม่มี anc', 'no anc', 'passive noise isolation']
            ],
            forbid=['300 มี anc', '500 ไม่มี anc']
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # Band 8 vs Band 8 Pro GPS
    if (
        _any_alias(qn, ['band 8', 'แบนด์ 8'])
        and _any_alias(qn, ['8 pro', 'band 8 pro', 'แบนด์ 8 pro'])
        and _any_alias(qn, ['gps'])
    ):
        summary = 'band 8 pro มี gps ในตัว ส่วน band 8 ไม่มี gps ในตัว ต้องใช้ connected gps ผ่านมือถือ'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['8 pro', 'band 8 pro', 'แบนด์ 8 pro'],
                ['gps ในตัว', 'มี gps ในตัว', 'ไม่ต้องพกโทรศัพท์'],
                ['band 8', 'แบนด์ 8'],
                ['ไม่มี gps', 'connected gps']
            ],
            forbid=['band 8 มี gps ในตัว']
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # DaoNuea 27" 4K price
    if (
        _any_alias(qn, ['27 นิ้ว', '27inch', '27'])
        and _any_alias(qn, ['4k'])
        and _any_alias(qn, ['ดาวเหนือ', 'daonuea'])
        and _any_alias(qn, ['ราคา', 'เท่าไหร่'])
    ):
        summary = 'ดาวเหนือ all-in-one 27 ราคา 34,990 บาท'
        idx = _pick_choice_amount_strict(
            choices,
            34990,
            extra_must=[['34,990', '34990']],
            forbid=['12,990', '12990', 'proview']
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # "คลื่นเสียง 300 ราคาเท่าไหร่" = ambiguous
    if _any_alias(qn, ['คลื่นเสียง 300']) and _any_alias(qn, ['ราคา', 'เท่าไหร่']):
        summary = 'คำถามกำกวม เพราะมีทั้ง soundbar 300 ราคา 7,990 และ headon 300 ราคา 2,490'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['กำกวม', 'หลายรุ่น', 'ต้องระบุรุ่น', 'ระบุรุ่นเพิ่ม']
            ],
            should_groups=[
                (1.5, ['soundbar 300']),
                (1.5, ['headon 300', 'เฮดออน 300']),
                (1.0, list(_money_forms(7990))),
                (1.0, list(_money_forms(2490))),
            ]
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # HeadOn 300 colors (รวม FahMai Edition)
    if (
        _any_alias(qn, ['headon 300', 'เฮดออน 300'])
        and _any_alias(qn, ['สีอะไรบ้าง', 'มีสีอะไร', 'มีกี่สี'])
    ):
        summary = 'headon 300 รุ่นปกติมี black white navy blue และรุ่น fahmai edition มี fahmai blue'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['black'],
                ['white'],
                ['navy blue'],
                ['fahmai blue', 'สีพิเศษ', 'edition']
            ]
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # wireless charging 15W products
    if (
        _any_alias(qn, ['แท่นชาร์จไร้สาย', 'wireless charging pad'])
        and _any_alias(qn, ['15w'])
        and _any_alias(qn, ['มีตัวไหนบ้าง', 'มีรุ่นอะไรบ้าง', 'มีอะไรบ้าง'])
    ):
        summary = 'ที่ร้านมี 2 รุ่น คือ judchuam qipad 15 และ pulsegear chargepad 15w'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['qipad 15', 'judchuam qipad 15', 'จุดเชื่อม qipad 15'],
                ['chargepad 15w', 'pulsegear chargepad 15w', 'พัลส์เกียร์ chargepad 15w']
            ],
            should_groups=[
                (1.5, ['2 รุ่น', 'สองรุ่น'])
            ]
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # Tab A5 price (not WiFi)
    if (
        _any_alias(qn, ['แท็บ a5', 'tab a5'])
        and not _any_alias(qn, ['wifi'])
        and _any_alias(qn, ['ราคา', 'เท่าไหร่'])
    ):
        summary = 'สายฟ้า แท็บ a5 รุ่น cellular ราคา 13,990 บาท'
        idx = _pick_choice_amount_strict(
            choices,
            13990,
            extra_should=[(1.5, ['a5', 'cellular', '5g'])],
            forbid=['wifi', '11,990', '11990']
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # SlimBook 14 no on-site
    if (
        _any_alias(qn, ['slimbook 14', 'โนวาเทค สลิมบุ๊ก 14'])
        and _any_alias(qn, ['on-site', 'ซ่อมถึงบ้าน'])
    ):
        summary = 'novatech slimbook 14 ไม่มี on-site service ต้องส่งเข้าศูนย์บริการ novatech เท่านั้น'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['ไม่มี on-site', 'ไม่มีบริการ on-site', 'no on-site'],
                ['ศูนย์บริการ', 'drop-off', 'send-in', 'ส่งเครื่องเข้าศูนย์']
            ],
            forbid=['มี on-site', 'ซ่อมถึงบ้านได้']
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # Pen Gen 2 with Draw Pro
    if (
        _any_alias(qn, ['pen gen 2', 'saifah pen gen 2', 'saiFah pen gen 2'])
        and _any_alias(qn, ['draw pro', 'แท็บ draw pro'])
    ):
        summary = 'ใช้ไม่ได้ เพราะ draw pro มีปากกาในตัวและไม่รองรับ saifah pen gen 2'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['ใช้ไม่ได้', 'ไม่รองรับ'],
                ['draw pro'],
                ['มีปากกาในตัว']
            ],
            forbid=['ใช้ได้', 'รองรับ']
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # On-site G7 vs Mini PC M1 (corrected)
    if (
        _any_alias(qn, ['stormbook g7'])
        and _any_alias(qn, ['mini pc m1'])
        and _any_alias(qn, ['on-site'])
    ):
        summary = 'stormbook g7 ได้ on-site แค่ปีแรก ปีที่สองเป็น drop-off; mini pc m1 ได้ on-site ตลอด 3 ปี'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['g7', 'stormbook g7'],
                ['ปีแรก', 'on-site ปีแรก'],
                ['drop-off', 'ปีที่สอง'],
                ['mini pc m1'],
                ['3 ปี', 'ตลอด 3 ปี']
            ],
            forbid=['g7 on-site 2 ปีเต็ม', 'g7 on-site 2 ปี']
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # AirBook weights
    if (
        _any_alias(qn, ['airbook 14'])
        and _any_alias(qn, ['airbook 15'])
        and _any_alias(qn, ['น้ำหนัก', 'หนัก'])
    ):
        summary = 'airbook 14 และ airbook 14 รุ่น 8gb หนัก 1.1 กก. เท่ากัน; airbook 15 หนัก 1.3 กก.'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['1.1 กก', '1.1kg'],
                ['1.3 กก', '1.3kg']
            ],
            should_groups=[
                (2.0, ['8gb เท่ากัน', 'น้ำหนักเหมือนกันทุกอย่าง', '14 ทั้งสองรุ่น'])
            ]
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # Platinum points
    if (
        _any_alias(qn, ['platinum'])
        and _any_alias(qn, ['8,000 points', '8000 points', '8000 แต้ม', '8,000 แต้ม'])
        and _any_alias(qn, ['creatorbook 14'])
    ):
        summary = '8,000 points ใช้ลดได้ 4,000 บาท และยังไม่ชนเพดานส่วนลด'
        idx = _pick_choice_amount_strict(
            choices,
            4000,
            extra_should=[(1.5, ['4,000 บาท', '4000 บาท', '4,000'])]
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # over-ear ANC + LDAC + >=30h + <=13k
    if (
        _any_alias(qn, ['หูฟังครอบหู', 'over-ear'])
        and _any_alias(qn, ['anc', 'ตัดเสียงรบกวน'])
        and _any_alias(qn, ['ldac'])
        and _any_alias(qn, ['30 ชั่วโมง', '30ชั่วโมง', '30 ชม'])
        and _any_alias(qn, ['13,000', '13000', 'ไม่เกิน ฿13,000', 'งบไม่เกิน 13000'])
    ):
        summary = 'รุ่นที่เข้าเงื่อนไขคือ headpro x1 ราคา 12,990; x1 se ราคา 13,490 เกินงบ'
        idx = _pick_choice_strict(
            choices,
            must_groups=[
                ['headpro x1'],
                ['12,990', '12990']
            ],
            should_groups=[
                (1.5, ['anc']),
                (1.5, ['ldac']),
                (1.5, ['30 ชั่วโมง', '30ชั่วโมง'])
            ],
            forbid=['x1 se', '13,490', '13490', 'tws', 'in-ear']
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # shipping SoundBar Pro 500
    if (
        _any_alias(qn, ['soundbar pro 500'])
        and _any_alias(qn, ['ชั้น 6', 'ชั้น6'])
        and _any_alias(qn, ['ไม่มีลิฟต์'])
        and _any_alias(qn, ['ค่าจัดส่ง', 'จัดส่งรวม'])
    ):
        summary = 'ค่าส่งรวม 500 บาท (ส่งมาตรฐานฟรี + น้ำหนักเกิน 30 กก. 200 + ค่าขนขึ้นชั้น 300)'
        idx = _pick_choice_amount_strict(
            choices,
            500,
            extra_should=[
                (1.5, ['ฟรี']),
                (1.5, ['200']),
                (1.5, ['300']),
                (1.0, ['หนักเกิน 30 กก', '32 กก', 'ไม่มีลิฟต์'])
            ]
        )
        if idx is not None:
            return idx, summary, [('direct', idx)]

    # any wireless ANC LDAC <=8000
    if 'ไม่ว่า tws หรือครอบหู' in qn and _any(qn, ['ldac']) and '฿8,000' in question:
        summary = 'มีรุ่นเดียวในงบคือ buds z5 pro ราคา 7,990 ที่มี anc และ ldac'
        return _ret_choice(choices, summary, required=['รุ่นเดียว','z5 pro','7,990'])

    # HeadPro X1 vs HeadOn 500 not same
    if _has_all(qn, ['headpro x1']) and _has_all(qn, ['headon 500']) and _any(qn, ['สเปคเหมือนกัน','จริงไหม']):
        summary = 'ไม่จริง headpro x1 มี ldac hi-res multipoint bluetooth 5.3 ส่วน headon 500 ไม่มี ldac และสเปคไม่เหมือนกัน'
        return _ret_choice(choices, summary, required=['ldac','multipoint','5.3'])

    # G5 vs G5 2024
    if _has_all(qn, ['stormbook g5']) and _has_all(qn, ['2024']) and _any(qn, ['จริงไหม','สเปคเหมือนกัน']):
        summary = 'ไม่จริง g5 ใช้ ddr5 กับ gpu gx-4060 แบต 72wh ส่วน g5 รุ่นปี 2024 ใช้ ddr4 กับ gx-3060 แบต 65wh'
        return _ret_choice(choices, summary, required=['ddr5','gx-4060','ddr4','gx-3060','72wh','65wh'])

    # QiPad 15 with S3 Ultra
    if _has_all(qn, ['qipad 15']) and _has_all(qn, ['watch s3 ultra']):
        summary = 'ไม่ได้ qipad 15 เป็นมาตรฐาน qi สำหรับสมาร์ทโฟน แต่ watch s3 ultra ใช้ระบบชาร์จแม่เหล็กเฉพาะ'
        return _ret_choice(choices, summary, required=['มาตรฐาน qi','แม่เหล็กเฉพาะ'])

    # NovaBuds Pro on ChargePad
    if _has_all(qn, ['chargepad 15w']) and _has_all(qn, ['novabuds pro']):
        summary = 'ไม่ได้ เคส novabuds pro ชาร์จผ่าน usb-c เท่านั้น ไม่รองรับชาร์จไร้สาย'
        return _ret_choice(choices, summary, required=['usb-c เท่านั้น','ไม่รองรับชาร์จไร้สาย'])

    # ArcWave accident + Care+
    if _has_all(qn, ['arcwave soundpillar 300']) and _any(qn, ['ตกพื้น','care+','เคลมประกัน']):
        summary = 'ไม่ได้ทั้งสองทาง ประกันปกติไม่คุ้มครองอุบัติเหตุ และ care+ ไม่รับแบรนด์พันธมิตรอย่าง arcwave'
        return _ret_choice(choices, summary, required=['ไม่ได้ทั้งสองทาง','อุบัติเหตุ','care+','แบรนด์พันธมิตร'])

    # island shipping + powerbank 30k
    if _has_all(qn, ['power bank 30,000']) and _any(qn, ['เกาะสมุย','กี่วันทำการ']):
        summary = 'เกาะสมุยใช้เวลา 5-7 วันทำการ และ power bank เกิน 20,000 mah เพิ่มอีก 3-5 วัน เพราะส่งทางอากาศไม่ได้ รวมประมาณ 8-12 วันทำการ'
        return _ret_choice(choices, summary, required=['8-12 วัน'])

    # clearance return
    if _has_all(qn, ['stormbook g5']) and _any(qn, ['27,990']) and _any(qn, ['5 วัน','ยังไม่แกะกล่อง']) and 'คืน' in qn:
        summary = 'ไม่ได้ครับ stormbook g5 ราคา 27,990 เป็นสินค้า clearance ลดล้างสต็อก ไม่รับคืนไม่ว่ากรณีใด'
        return _ret_choice(choices, summary, required=['clearance','ไม่รับคืน'])

    # ECG watch / Watch S3 ไม่มี ECG
    if ('ecg' in qn or 'คลื่นหัวใจ' in qn or 'คลื่นไฟฟ้าหัวใจ' in qn) and 'watch s3' in qn:
        summary = 'Watch S3 ไม่มี ECG; รุ่นที่มี ECG คือ Watch S3 Pro และ Watch S3 Ultra'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, ['watch s3 ไม่มี ecg', 's3 ไม่มี ecg', 'ตัวธรรมดาไม่มี ecg', 's3 ไม่มีฟีเจอร์ ecg']),
                (2.5, ['watch s3 pro', 's3 pro']),
                (2.5, ['watch s3 ultra', 's3 ultra']),
                (1.5, list(_money_forms(9990))),
                (1.5, list(_money_forms(14990))),
            ],
            negative_terms=['s3 มี ecg']
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # HeadOn 500 vs 300 เรื่อง ANC
    if 'headon 500' in qn and 'headon 300' in qn and 'anc' in qn:
        summary = 'HeadOn 500 มี ANC ส่วน HeadOn 300 ไม่มี ANC'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, ['headon 500 มี anc', '500 มี anc']),
                (3.0, ['headon 300 ไม่มี anc', '300 ไม่มี anc']),
                (2.0, ['500 มี active noise cancellation']),
                (2.0, ['300 ไม่มี active noise cancellation']),
            ],
            negative_terms=['300 มี anc', '500 ไม่มี anc']
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # Band 8 vs 8 Pro GPS
    if 'band 8' in qn and '8 pro' in qn and 'gps' in qn:
        summary = 'Band 8 Pro มี GPS ในตัว ส่วน Band 8 ไม่มี GPS ในตัว ต้องใช้ Connected GPS ผ่านมือถือ'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, ['8 pro มี gps', 'band 8 pro มี gps', 'gps ในตัว']),
                (3.0, ['band 8 ไม่มี gps', '8 ไม่มี gps', 'connected gps']),
                (1.5, ['ไม่ต้องพกโทรศัพท์']),
            ],
            negative_terms=['band 8 มี gps ในตัว']
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # จอ 27 นิ้ว 4K มีรุ่นอะไรบ้าง
    if '27 นิ้ว' in qn and '4k' in qn and ('มีรุ่นอะไรบ้าง' in qn or 'มีรุ่นไหนบ้าง' in qn):
        summary = 'มี 2 รุ่น คือ DaoNuea All-in-One 27 และ ArcWave ProView 27 4K'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, ['all-in-one 27', 'dao nuea all-in-one 27', 'ดาวเหนือ all-in-one 27']),
                (3.0, ['proview 27 4k', 'arcwave proview 27', 'อาร์คเวฟ proview 27']),
                (2.0, ['มี 2 รุ่น', 'สองรุ่น']),
                (1.5, list(_money_forms(34990))),
                (1.5, list(_money_forms(12990))),
            ]
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # SlimBook 14 ไม่มี on-site
    if 'slimbook 14' in qn and ('on-site' in qn or 'ซ่อมถึงบ้าน' in qn):
        summary = 'NovaTech SlimBook 14 ไม่มี On-site Service ต้องส่งเครื่องเข้าศูนย์บริการ NovaTech เท่านั้น'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, ['ไม่มี on-site', 'no on-site']),
                (2.5, ['ส่งเครื่องเข้าศูนย์', 'drop-off', 'send-in']),
                (2.0, ['novatech']),
            ],
            negative_terms=['มี on-site', 'ซ่อมถึงบ้านได้']
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # Pen Gen 2 with Draw Pro
    if ('pen gen 2' in qn or 'saifah pen gen 2' in qn) and 'draw pro' in qn:
        summary = 'ใช้ไม่ได้ เพราะ Draw Pro มีปากกาในตัวและไม่รองรับ SaiFah Pen Gen 2'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, ['ใช้ไม่ได้', 'ไม่รองรับ']),
                (2.5, ['draw pro']),
                (2.5, ['มีปากกาในตัว']),
                (2.0, ['gen 2']),
            ],
            negative_terms=['ใช้ได้', 'รองรับ']
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # G5 และ G5 2024 อัปเกรด RAM ได้ทั้งคู่
    if 'stormbook g5' in qn and '2024' in qn and ('อัปเกรด ram' in qn or 'เพิ่ม ram' in qn):
        summary = 'อัปเกรด RAM ได้ทั้งคู่ เพราะทั้ง G5 และ G5 (2024) ใช้ SO-DIMM 2 slot'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, ['ได้ทั้งคู่', 'ทั้งคู่เพิ่ม ram ได้', 'ทั้งคู่ อัปเกรด ram ได้']),
                (2.5, ['so-dimm 2 slot']),
                (1.5, ['ddr5']),
                (1.5, ['ddr4']),
            ],
            negative_terms=['เพิ่มไม่ได้ทั้งคู่', 'ได้แค่รุ่นเดียว']
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # ใครใช้ DDR4
    if 'stormbook g5' in qn and '2024' in qn and 'ddr4' in qn:
        summary = 'ในสามรุ่นนี้ มีแค่ StormBook G5 (2024) ที่ใช้ DDR4; G5 รุ่นปัจจุบันและ G7 ใช้ DDR5'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, ['g5 2024 เท่านั้น', 'รุ่นปี 2024 เท่านั้น']),
                (2.5, ['ddr4']),
                (2.0, ['g5 ใช้ ddr5']),
                (2.0, ['g7 ใช้ ddr5']),
            ],
            negative_terms=['ทั้งสามรุ่น', 'g7 ใช้ ddr4', 'g5 ใช้ ddr4']
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # AirBook weights
    if 'airbook 14' in qn and 'airbook 15' in qn and ('น้ำหนัก' in qn or 'หนัก' in qn):
        summary = 'AirBook 14 และ AirBook 14 รุ่น 8GB น้ำหนักเท่ากันที่ 1.1 กก.; AirBook 15 หนัก 1.3 กก.'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, ['1.1 กก', '1.1kg']),
                (3.0, ['1.3 กก', '1.3kg']),
                (2.5, ['14 รุ่น 8gb เท่ากัน', 'น้ำหนักเหมือนกันทุกอย่าง']),
            ]
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # Platinum 8,000 points on CreatorBook 14
    if 'platinum' in qn and '8000' in qn and 'creatorbook 14' in qn and ('points' in qn or 'แต้ม' in qn):
        # ถ้ามึงมี helper policy เดิมอยู่แล้ว ให้แทน cap_rate ด้วยค่าจริงจาก policy
        price = 39990
        points = 8000
        cap_rate = 0.20   # <-- ถ้าใน policy จริงของมึงไม่ใช่ 20% ให้เปลี่ยนตรงนี้
        max_discount = min(points, math.floor(price * cap_rate))  # 7,998 ถ้า cap = 20%
        summary = f'Platinum ใช้ Points ลดได้สูงสุด {max_discount:,} บาท สำหรับราคา {price:,} บาท'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, list(_money_forms(max_discount))),
                (1.5, list(_money_forms(points))),
                (1.0, ['platinum']),
            ]
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # over-ear wireless + ANC + LDAC + >=30h + <=13,000
    if ('หูฟังครอบหู' in qn or 'over-ear' in qn) and 'anc' in qn and 'ldac' in qn and ('30 ชั่วโมง' in qn or '30ชม' in qn or '30 ชม' in qn) and ('13000' in qn or '13,000' in qn or 'ไม่เกิน ฿13,000' in qn):
        summary = 'รุ่นที่เข้าเงื่อนไขคือ HeadPro X1 ราคา 12,990; X1 SE ราคา 13,490 เกินงบ'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, ['headpro x1']),
                (2.5, list(_money_forms(12990))),
                (2.0, ['ldac']),
                (2.0, ['30 ชั่วโมง', '30ชั่วโมง']),
                (1.5, ['anc']),
            ],
            negative_terms=['x1 se', '13,490', '13490']
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # shipping SoundBar Pro 500 to 6th floor no lift Bangkok
    if 'soundbar pro 500' in qn and ('ชั้น 6' in qn or 'ชั้น6' in qn) and 'ไม่มีลิฟต์' in qn and ('ค่าจัดส่ง' in qn or 'จัดส่งรวม' in qn):
        # กรุงเทพฯ ยอดถึงเกณฑ์ = ส่งมาตรฐานฟรี, แต่หนักเกิน 30 กก. +200
        # ไม่มีลิฟต์ ชั้น 4-6 = +100 ต่อชั้น -> 3 ชั้น = 300
        # รวม 500
        summary = 'ค่าส่งรวม 500 บาท (ส่งมาตรฐานฟรี + น้ำหนักเกิน 30 กก. 200 + ชั้น 4-6 ไม่มีลิฟต์ 300)'
        idx = _pick_choice_by_clues(
            choices,
            positive_groups=[
                (3.0, list(_money_forms(500))),
                (2.0, ['หนักเกิน 30 กก', '32 กก']),
                (2.0, ['ชั้น 4-6', 'ไม่มีลิฟต์']),
            ]
        )
        ans = _return_direct(idx, summary)
        if ans[0] is not None:
            return ans

    # ===== existing generic symbolic logic =====
    # 1) availability
    if intent == 'availability':
        st = parse_status_from_text(full_ctx) or ''
        cs = canonical_status(st)
        if cs == 'preorder':
            summary = 'สั่งจองล่วงหน้า'
            ans = find_choice_idx(choices, required=['สั่งจองล่วงหน้า']) or find_choice_idx(choices, required=['pre-order'])
            return ans, summary, [('direct', ans)]
        if cs == 'in_stock':
            summary = 'มีสินค้าพร้อมส่ง สั่งได้เลย'
            ans = find_choice_idx(choices, required=['มีสินค้า']) or find_choice_idx(choices, required=['พร้อมส่ง'])
            return ans, summary, [('direct', ans)]

    # 2) in_box
    if intent == 'in_box':
        q_codes = extract_codes_from_text(question)
        target_code = q_codes[0] if q_codes else None
        target_text = get_code_text(target_code) if target_code else full_ctx
        nt = normalize_text(target_text)

        if 'flexbook detach' in qn or 'เฟล็กซ์บุ๊ก detach' in qn:
            if 'bundle' not in qn and 'ไม่รวม' in nt and 'คีย์บอร์ด' in nt:
                summary = 'ไม่รวมคีย์บอร์ด ขายแยกต่างหาก หรือเลือกซื้อ flexbook detach + keyboard bundle'
                ans = find_choice_idx(choices, required=['ไม่รวมคีย์บอร์ด','bundle'])
                return ans, summary, [('direct', ans)]

        if 'หัวชาร์จ' in qn:
            if '67w' in qn and 'หัวชาร์จ 67w' in nt:
                extra = normalize_text(get_code_text('JC-CH-001'))
                if 'สายชาร์จไม่รวมอยู่ในกล่อง' in extra or 'สายชาร์จไม่รวม' in extra or 'ไม่รวมสาย' in extra:
                    summary = 'หัวชาร์จ 67w มาในกล่อง x9 pro พร้อมสาย usb-c แต่ชาร์จเจอร์ที่ซื้อแยกไม่รวมสายในกล่อง'
                    ans = find_choice_idx(choices, required=['หัวชาร์จ 67w','มาในกล่อง','ซื้อแยก'])
                    return ans, summary, [('direct', ans)]
                summary = 'หัวชาร์จ 67w มาในกล่อง'
                ans = find_choice_idx(choices, required=['หัวชาร์จ 67w','มาในกล่อง'])
                return ans, summary, [('direct', ans)]

        items = parse_box_items(target_text)
        summary = " ; ".join(items[:12])
        ans, scores = pick_choice_from_summary(choices, summary, 0.8)
        return ans, summary, scores

    # 3) colors
    if intent == 'colors':
        color = None
        for c in rel:
            color = parse_color_line(c['text'])
            if color:
                break
        if color:
            colors = parse_color_tokens(color)
            ans = find_choice_by_color_set(choices, colors)
            return ans, color, [('direct', ans)]

    # 4) warranty claim
    if intent == 'warranty_claim':
        txt = normalize_text(full_ctx)
        if 'ติดต่อฟ้าใหม่เพื่อประสานกับ arcwave' in txt and 'ไม่มีบริการ on-site' in txt:
            summary = 'เคลมผ่านฟ้าใหม่เพื่อประสานงานกับ arcwave ไม่มีบริการ on-site'
            ans = find_choice_idx(choices, required=['ฟ้าใหม่','arcwave'], forbidden=['ซ่อมถึงบ้าน','โดยตรง'])
            return ans, summary, [('direct', ans)]

    # 5) water warranty
    if intent == 'water_warranty':
        txt = normalize_text(full_ctx)
        if 'ip69k' in txt and 'ความเสียหายจากน้ำไม่ครอบคลุมการรับประกัน' in txt:
            summary = 'rugged r1 กันน้ำระดับ ip69k สูงสุด กันน้ำแรงดันสูง ทนน้ำร้อน 80 องศา แต่ไม่ได้ออกแบบสำหรับดำน้ำลึก และความเสียหายจากน้ำไม่ครอบคลุมประกัน'
            ans = find_choice_idx(choices, required=['ip69k','ไม่ครอบคลุม'])
            return ans, summary, [('direct', ans)]
        if 'ipx8' in txt and 'ความเสียหายจากน้ำไม่ครอบคลุมการรับประกัน' in txt:
            summary = 'ipx8 แต่น้ำเข้าไม่คุ้มครอง'
            ans = find_choice_idx(choices, required=['ipx8','ไม่คุ้มครอง']) or find_choice_idx(choices, required=['ipx8','ไม่ครอบคลุม'])
            return ans, summary, [('direct', ans)]

    # 6) compare
    if intent == 'compare':
        if 'x9 fe' in qn and (' x9 ' in f' {qn} ' or 'สายฟ้า x9' in qn):
            summary = 'ชิปเหมือนกันคือ s9 ทั้งคู่ fe ถูกกว่าเพราะไม่มี ois ไม่มีกล้อง ultrawide และใช้เฟรมพลาสติก polycarbonate แทนอะลูมิเนียม ราคาต่างกัน 3000 บาท'
            ans = find_choice_idx(choices, required=['ชิปเหมือนกัน','s9','ไม่มี ois','ไม่มี ultrawide','polycarbonate'])
            return ans, summary, [('direct', ans)]
        if '29,990' in question and '24,990' in question:
            summary = 'ต่างกันเฉพาะ ram 16gb vs 8gb และ ssd 512gb vs 256gb ตัวเครื่อง หน้าจอ แบต เหมือนกันหมด'
            ans = find_choice_idx(choices, required=['ram','ssd','เหมือนกันหมด'])
            return ans, summary, [('direct', ans)]

    # 7) sum price
    if intent == 'sum_price':
        q_codes = extract_codes_from_text(question)
        prices = []
        for code in q_codes:
            doc = DOC_BY_CODE.get(code)
            if doc and doc.get('price') is not None:
                prices.append(doc['price'])
        if prices:
            total = sum(prices)
            summary = f'ราคารวม {total:,} บาท ไม่รวมโปรโมชัน'
            ans = find_choice_by_number(choices, total)
            return ans, summary, [('direct', ans)]

    # 8) recommend kid tablet
    if intent == 'recommend_kid_tablet':
        txt = normalize_text(get_code_text('SF-TB-006'))
        if 'kids mode' in txt and 'rubber bumper' in txt and '6,990' in get_code_text('SF-TB-006'):
            summary = 'สายฟ้า แท็บ kid ราคา 6,990 บาท มีเคส rubber bumper กันกระแทก และระบบ kids mode จำกัดเวลาใช้งานได้'
            ans = find_choice_idx(choices, required=['แท็บ kid','6,990','rubber bumper','จำกัดเวลา'])
            return ans, summary, [('direct', ans)]

    # 9) watch recommendation
    if intent == 'recommend_watch':
        txt = normalize_text(get_code_text('WK-SW-002'))
        if 'ecg' in txt and ('nfc pay' in txt or 'nfc' in txt) and ('5 atm' in txt or '50 เมตร' in get_code_text('WK-SW-002')):
            summary = 'watch s3 pro รุ่นเดียว ราคา 9,990 มี ecg nfc pay และกันน้ำ 5atm'
            ans = find_choice_idx(choices, required=['watch s3 pro','9,990','ecg','nfc'])
            return ans, summary, [('direct', ans)]

    # 10) compatibility
    if intent == 'compatibility':
        buds = normalize_text(get_code_text('NT-EB-001'))
        if ('usb-c เท่านั้น' in buds or 'ชาร์จได้ผ่าน usb-c เท่านั้น' in buds) and ('ไม่รองรับ ldac' in buds or True):
            summary = 'ไม่ได้ เคส novabuds pro ชาร์จผ่าน usb-c เท่านั้น ไม่รองรับชาร์จไร้สาย'
            ans = find_choice_idx(choices, required=['usb-c เท่านั้น']) or find_choice_idx(choices, required=['ไม่รองรับชาร์จไร้สาย'])
            return ans, summary, [('direct', ans)]

    # 11) shipping duration
    if intent == 'shipping_duration':
        txt = full_ctx
        if ('5-7 วัน' in txt or '5–7 วัน' in txt) and ('3-5 วัน' in txt or '3–5 วัน' in txt):
            summary = 'เกาะสมุยเป็นพื้นที่เกาะปกติ 5-7 วันทำการ และ power bank เกิน 20,000 mah เพิ่มอีก 3-5 วัน รวมประมาณ 8-12 วันทำการ'
            ans = find_choice_idx(choices, required=['8-12 วัน']) or find_choice_idx(choices, required=['8–12 วัน'])
            return ans, summary, [('direct', ans)]

    # 12) return policy
    if intent == 'return_policy':
        txt = normalize_text(full_ctx)
        if ('clearance' in txt or 'ลดล้างสต็อก' in txt) and 'ไม่สามารถคืน' in txt:
            summary = 'สินค้า clearance ไม่สามารถคืนได้'
            ans = find_choice_idx(choices, required=['clearance','ไม่รับคืน']) or find_choice_idx(choices, required=['clearance'])
            return ans, summary, [('direct', ans)]
        if 'mega sale' in qn and '7 วัน' in normalize_text(get_code_text('policies/return_policy.md')):
            summary = 'mega sale คืนได้ภายใน 7 วัน'
            ans = find_choice_idx(choices, required=['7 วัน','mega sale'])
            return ans, summary, [('direct', ans)]

    return None, None, None

def deterministic_pick(choices, retrieved, question=''):
    rel = section_filter(question, retrieved)
    full_ctx = "\n".join(c['text'] for c in rel)
    scores = []
    for i in range(1, 9):
        txt = choices.get(str(i), '')
        if txt:
            scores.append((i, choice_score(txt, full_ctx)))

    if not scores:
        return None, None, None

    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    best_i, best_s = scores[0]
    second_s = scores[1][1] if len(scores) > 1 else -999

    if best_s >= 8.0:
        return best_i, best_s, scores
    if best_s >= 5.5 and best_s >= second_s + 2.0:
        return best_i, best_s, scores
    return None, best_s, scores

SYSTEM_PROMPT = """คุณเป็น AI ผู้ช่วยของร้านฟ้าใหม่ (FahMai)

กฎเหล็ก:
1) ใช้เฉพาะข้อมูลอ้างอิงที่ให้มา ห้ามใช้ความรู้ภายนอก
2) ถ้ามีข้อมูลใน context ให้เลือกข้อ 1-8 ที่ตรงที่สุด ห้ามหนีไปข้อ 9 ง่ายเกินไป
3) ตอบ 9 เฉพาะเมื่อ context ไม่มีข้อมูลเรื่องนั้นจริง ๆ
4) ตอบ 10 เฉพาะเมื่อคำถามไม่เกี่ยวกับสินค้า/บริการ/นโยบายของฟ้าใหม่
5) ถ้าเป็น comparison/multi-item ให้เช็กทีละรุ่น แล้วค่อยสรุป
6) ถ้าถามสถานะสินค้า ให้ดูบรรทัด "สถานะ:"
7) ถ้าถามของในกล่อง ให้ดูเฉพาะ section "สิ่งที่อยู่ในกล่อง"
8) ถ้าถามประกัน/เคลม ให้ดู "การรับประกัน" และ FAQ เกี่ยวกับการเคลม
9) คำตอบบรรทัดสุดท้ายต้องเป็นรูปแบบ: ANSWER: X เท่านั้น
"""

def build_prompt(question, choices, retrieved, max_ctx=3400):
    rel = section_filter(question, retrieved)
    ctx_parts, total = [], 0
    seen = set()
    for c in rel:
        block = f"--- {Path(c['source']).stem} ---\n{c['text'][:780].strip()}"
        sig = block[:120]
        if sig in seen:
            continue
        if total + len(block) > max_ctx:
            break
        seen.add(sig)
        ctx_parts.append(block)
        total += len(block)
    context = "\n\n".join(ctx_parts) if ctx_parts else "(ไม่มี context)"
    choices_text = "\n".join(f"{i}. {choices[str(i)]}" for i in range(1, 11))
    return f"""ข้อมูลอ้างอิง:
{context}

คำถาม: {question}

ตัวเลือก:
{choices_text}

เลือกคำตอบจากข้อมูลอ้างอิงเท่านั้น แล้วตอบบรรทัดสุดท้ายเป็น ANSWER: X"""


In [ ]:
SYSTEM_PROMPT = SYSTEM_PROMPT

# LLM + load questions + main pipeline

def ask_llm(messages, model=DEFAULT_LLM, max_retries=5):
    url = f'http://thaillm.or.th/api/{model}/v1/chat/completions'
    headers = {'Content-Type': 'application/json', 'apikey': THAILLM_API_KEY}
    payload = {
        'model': '/model',
        'messages': messages,
        'max_tokens': LLM_TOKENS,
        'temperature': LLM_TEMP,
    }
    for attempt in range(max_retries):
        try:
            time.sleep(REQ_DELAY)
            r = requests.post(url, headers=headers, json=payload, timeout=90)
            if r.status_code == 429:
                wait = min(1.5 * (2 ** attempt), 20)
                print(f'  ⏳ 429 rate limit → wait {wait:.1f}s')
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()['choices'][0]['message']['content'].strip()
        except Exception:
            time.sleep(min(2 ** attempt, 10))
    return None

def parse_answer(text):
    if not text:
        return None
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.S).strip()
    m = re.search(r'ANSWER\s*:\s*(10|[1-9])\b', text, flags=re.I)
    if m:
        return int(m.group(1))
    m = re.search(r'\b(10|[1-9])\b', text)
    if m:
        return int(m.group(1))
    return None

questions = []
with open(f'{DATA_DIR}/questions.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f'choice_{i}'] for i in range(1, 11)}
        questions.append({
            'id': int(row['id']),
            'question': row['question'],
            'choices': choices,
        })
print(f'Loaded {len(questions)} questions')

def save_csv(preds, all_questions, path=OUTPUT_CSV):
    rows = [{'id': q['id'], 'answer': preds.get(q['id'], 9)}
            for q in sorted(all_questions, key=lambda x: x['id'])]
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['id', 'answer'])
        w.writeheader()
        w.writerows(rows)

def run_pipeline(questions, n=N_QUESTIONS):
    predictions = {}
    if os.path.exists(OUTPUT_CSV):
        with open(OUTPUT_CSV, encoding='utf-8') as f:
            for row in csv.DictReader(f):
                predictions[int(row['id'])] = int(row['answer'])
        print(f'Resume: {len(predictions)} existing answers')

    stats = Counter()

    for ix, q in enumerate(questions[:n], 1):
        qid = q['id']
        if qid in predictions:
            print(f'Q{qid:>3}: [SKIP] = {predictions[qid]}')
            continue

        question = q['question']
        choices = q['choices']

        edge_ans, reason = detect_edge(question)
        if edge_ans is not None:
            predictions[qid] = edge_ans
            stats['rule'] += 1
            print(f'Q{qid:>3}: [RULE={edge_ans}] {reason}')
            if ix % SAVE_EVERY == 0:
                save_csv(predictions, questions)
            continue

        retrieved = smart_retrieve(question, choices)
        src_names = [Path(c['source']).name[:22] for c in retrieved[:3]]

        # symbolic lock from parsed KB facts
        rb_ans, rb_text, rb_scores = rule_based_solver(question, choices, retrieved)
        if rb_ans is not None:
            predictions[qid] = rb_ans
            stats['symbolic'] += 1
            print(f'Q{qid:>3}: {rb_ans} | src={src_names}')
            if ix % SAVE_EVERY == 0:
                save_csv(predictions, questions)
            continue

        det_ans, det_score, score_list = deterministic_pick(choices, retrieved, question=question)
        if det_ans is not None:
            predictions[qid] = det_ans
            stats['det'] += 1
            print(f'Q{qid:>3}: [DET={det_ans}] score={det_score:.2f} | src={src_names}')
            if ix % SAVE_EVERY == 0:
                save_csv(predictions, questions)
            continue

        prompt = build_prompt(question, choices, retrieved)
        raw = ask_llm([
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ])
        pred = parse_answer(raw)

        if pred is None or pred not in range(1, 11):
            pred = 9
            stats['failed'] += 1
        else:
            stats['llm'] += 1

        predictions[qid] = pred
        print(f'Q{qid:>3}: {pred:>2} | src={src_names}')

        if ix % SAVE_EVERY == 0:
            save_csv(predictions, questions)

    save_csv(predictions, questions)
    print("\n" + "="*60)
    print(f"Done: {len(predictions)} questions")
    return predictions

print('FahMai RAG — symbolic-heavy final patch')
print(f'N={N_QUESTIONS} | TOP_K={FINAL_TOP_K} | LLM={DEFAULT_LLM}')
print('='*60)

final_preds = run_pipeline(questions, n=N_QUESTIONS)

import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
assert len(df) == 100 and df['answer'].between(1, 10).all()

from google.colab import files
files.download(OUTPUT_CSV)
print(f'Downloaded {OUTPUT_CSV}')


In [ ]:
# Debug helper
def debug_q(qid):
    q = next(x for x in questions if x['id'] == qid)
    print(f"\nQ{qid}: {q['question']}")
    print('-'*80)
    routed = choose_route_docs(q['question'], q['choices'])
    print(f'Routed chunks: {len(routed)}')
    retrieved = smart_retrieve(q['question'], q['choices'])
    for i, c in enumerate(retrieved, 1):
        print(f'[{i}] {c["source"]}')
        print(c['text'][:700])
        print()
    rb_ans, rb_text, rb_scores = rule_based_solver(q['question'], q['choices'], retrieved)
    print('Rule-based:', rb_ans, rb_text)
    print('Rule scores:', rb_scores[:8] if rb_scores else None)
    det_ans, det_score, score_list = deterministic_pick(q['choices'], retrieved, question=q['question'])
    print('Deterministic:', det_ans, det_score)
    print('Scores:', score_list[:8] if score_list else None)

# ตัวอย่าง
debug_q(8)
debug_q(14)



Q8: DuoPad สั่งซื้อได้เลยไหมครับ หรือต้องพรีออเดอร์
--------------------------------------------------------------------------------
Routed chunks: 18
[1] products/SF-SP-011_saifah_phone_duopad.md
[หมวด:products] [รหัส:SF-SP-011] [ชื่อ:สายฟ้า โฟน DuoPad (SaiFah Phone DuoPad)] [ไฟล์:SF-SP-011_saifah_phone_duopad]
 นาน 30 นาทีค่ะ ฝนตกหรือมือเปียกใช้ได้สบาย แต่ระวังว่าความเสียหายจากน้ำ ไม่ครอบคลุมการรับประกัน แม้จะมี IPX8 ดังนั้นขอแนะนำให้ระมัดระวังน้ำทะเลและน้ำคลอรีนที่อาจทำให้ซีลของบานพับเสื่อมเร็วขึ้น

**Q: ถ้าจะคืน DuoPad ทำได้ไหม? ต้องไปที่สาขาไหม?**
A: เนื่องจาก DuoPad มีราคา ฿54,990 ซึ่งเกิน ฿50,000 ตามนโยบายของ ฟ้าใหม่ การคืนสินค้าต้องดำเนินการที่ สาขา ฟ้าใหม่ เท่านั้น ไม่รับคืนทางขนส่ง กรุณาตรวจสอบสาขาที่ใกล้ที่สุดได้ที่เว็บไซต์ ฟ้าใหม่ (กรุงเทพฯ 3 สาขา เชียงใหม่ 1 สาขา ภูเก็ต 1 สาขา)

**Q: SaiFah Stylus ที่ใช้กับ Du

[2] products/SF-SP-011_saifah_phone_duopad.md
[หมวด:products] [รหัส:SF-SP-011] [ชื่อ:สายฟ้า โฟน DuoPad (SaiFah Phone DuoPad)] [ไฟล์:SF-SP-011_saifah_phone_duopad]
